# MODUL PRAKTIKUM BIG DATA
## Pertemuan 3 — Pengenalan Big Data & Instalasi Apache Hadoop (Single-Node)

| | |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Pertemuan** | 3 |
| **Topik** | Konsep Dasar Big Data, Hadoop Ecosystem, Instalasi & Konfigurasi Apache Hadoop (mode Pseudo-Distributed) |
| **Estimasi Waktu** | 3 x 50 menit |
| **Prasyarat** | Telah menyelesaikan **Modul Pertemuan 1 & 2** (VM Ubuntu aktif, Anaconda + environment `bigdata` terpasang, folder `~/praktikum-bigdata` sudah ada) |

---

> **Konsistensi versi:** Modul ini menggunakan **Apache Hadoop 3.4.3** (rilis stabil terbaru pada jalur 3.4, per pertengahan 2026) dipasangkan dengan **OpenJDK 11**. **Jangan mengganti versi Java atau Hadoop secara sembarangan.**


---
## Recap Pertemuan Sebelumnya

Sebelum melanjutkan, pastikan hal-hal berikut dari Pertemuan 1 & 2 masih berfungsi:

- [ ] VM **Ubuntu-BigData** dapat dinyalakan dan login normal
- [ ] Perintah `conda activate bigdata` berhasil tanpa error
- [ ] Folder `~/praktikum-bigdata` masih ada, berisi modul-modul sebelumnya
- [ ] Jupyter Notebook dapat dijalankan dengan `jupyter notebook`

Jika salah satu di atas gagal, kembali ke Modul Pertemuan 1 & 2 untuk memperbaikinya terlebih dahulu sebelum melanjutkan.

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 3, mahasiswa mampu:
1. Menjelaskan konsep dasar Big Data (5V) dan mengapa Hadoop diciptakan untuk mengatasinya.
2. Menjelaskan peran komponen utama Hadoop Ecosystem: **HDFS**, **YARN**, dan **MapReduce**.
3. Menginstall dan mengkonfigurasi Apache Hadoop dalam mode *pseudo-distributed* (single-node) di Ubuntu.
4. Menjalankan dan memverifikasi seluruh proses (daemon) Hadoop.
5. Mengoperasikan perintah dasar HDFS untuk mengelola berkas pada sistem berkas terdistribusi.


---
## 3.1 Konsep Dasar Big Data

**Big Data** merujuk pada data yang volumenya, kecepatan pertumbuhannya, atau keragamannya sudah melampaui kemampuan sistem pengolah data konvensional (seperti satu server database biasa) untuk menampung dan memprosesnya. Karakteristik Big Data umumnya dijelaskan melalui kerangka **5V**:

| Karakteristik | Penjelasan | Contoh |
|---|---|---|
| **Volume** | Jumlah data yang sangat besar (terabyte hingga exabyte) | Data transaksi seluruh cabang toko retail nasional per hari |
| **Velocity** | Kecepatan data dihasilkan dan perlu diproses | Data sensor IoT pabrik yang mengirim pembacaan setiap detik |
| **Variety** | Beragam format data — terstruktur, semi-terstruktur, tak terstruktur | Tabel transaksi (terstruktur), log JSON (semi-terstruktur), foto/video (tak terstruktur) |
| **Veracity** | Tingkat keakuratan dan kepercayaan terhadap data | Data sensor yang bisa jadi *noisy* atau tidak lengkap |
| **Value** | Nilai/wawasan bisnis yang dapat digali dari data | Pola pembelian pelanggan untuk strategi promosi |

**Mengapa pandas saja tidak cukup?** Pada Pertemuan 2, kita mengolah data dengan pandas — namun pandas memuat **seluruh data ke dalam memori (RAM)** satu komputer. Ketika volume data mencapai ratusan GB hingga TB, atau ketika data tersebar di ribuan berkas pada banyak server, pendekatan satu-komputer seperti ini tidak lagi memadai. Di sinilah **Apache Hadoop** berperan — sebuah framework yang memungkinkan penyimpanan dan pemrosesan data secara **terdistribusi** di banyak komputer (disebut *cluster*) sekaligus.

---

## 3.2 Pengenalan Hadoop Ecosystem

Apache Hadoop terdiri dari tiga komponen inti yang saling melengkapi:

| Komponen | Kepanjangan | Fungsi |
|---|---|---|
| **HDFS** | Hadoop Distributed File System | Lapisan **penyimpanan**. Memecah berkas besar menjadi blok-blok, lalu menyebarkannya (dengan replikasi) ke banyak node agar tahan terhadap kegagalan satu mesin. |
| **YARN** | Yet Another Resource Negotiator | Lapisan **manajemen sumber daya**. Mengatur alokasi CPU/memori cluster untuk berbagai aplikasi yang berjalan di atasnya. |
| **MapReduce** | — | Lapisan **pemrosesan**. Model pemrograman untuk mengolah data besar secara paralel, dipecah menjadi tahap *Map* (memproses potongan data) dan *Reduce* (menggabungkan hasil). |

**Arsitektur HDFS secara ringkas:**
- **NameNode** — "otak" HDFS, menyimpan *metadata* (struktur folder, lokasi setiap blok data) — hanya ada satu NameNode aktif per cluster.
- **DataNode** — node pekerja yang benar-benar menyimpan blok-blok data fisik — sebuah cluster produksi memiliki banyak DataNode.
- **SecondaryNameNode** — membantu NameNode menggabungkan log perubahan secara berkala (bukan *backup* NameNode, sering disalahpahami).

> Pada praktikum ini kita menjalankan Hadoop dalam **mode pseudo-distributed** — seluruh proses (NameNode, DataNode, dst.) berjalan di **satu mesin** (VM kita), namun tetap dikonfigurasi persis seperti cluster sungguhan. Ini adalah cara standar untuk *belajar* Hadoop sebelum nanti bekerja dengan cluster multi-node yang sesungguhnya.


---
## 3.3 Instalasi Java (OpenJDK 11)

Hadoop dibangun di atas Java, sehingga JDK (*Java Development Kit*) wajib terpasang lebih dulu. Kita menggunakan **OpenJDK 11** — versi yang didukung penuh dan paling banyak digunakan bersama Hadoop 3.3.x/3.4.x.

**Buka Terminal, lalu jalankan:**

```bash
sudo apt update
sudo apt install openjdk-11-jdk -y
```

Verifikasi instalasi:

```bash
java -version
```

Output yang diharapkan (angka versi minor dapat sedikit berbeda):

```
openjdk version "11.0.XX" 2026-XX-XX
OpenJDK Runtime Environment (build 11.0.XX+X-Ubuntu-XubuntuX)
OpenJDK 64-Bit Server VM (build 11.0.XX+X-Ubuntu-XubuntuX, mixed mode, sharing)
```

Catat lokasi instalasi Java — kita akan membutuhkannya pada konfigurasi Hadoop nanti:

```bash
readlink -f $(which java)
```

Pada Ubuntu, hasilnya biasanya:

```
/usr/lib/jvm/java-11-openjdk-amd64/bin/java
```

Sehingga `JAVA_HOME` yang akan kita gunakan adalah **`/usr/lib/jvm/java-11-openjdk-amd64`** (tanpa `/bin/java` di akhir).

---

## 3.4 Konfigurasi SSH Tanpa Password

Meski berjalan dalam satu mesin, Hadoop tetap berkomunikasi antar-daemon melalui protokol SSH — sehingga SSH ke `localhost` **tanpa perlu memasukkan password** wajib dikonfigurasi, jika tidak, proses start/stop Hadoop akan macet menunggu input password berulang kali.

**Jalankan langkah berikut di Terminal:**

```bash
sudo apt install openssh-server openssh-client -y
```

Buat pasangan(pair) kunci SSH (SSH Key) (tekan **Enter** tiga kali saat diminta lokasi & passphrase, biarkan semuanya kosong/default):

```bash
ssh-keygen -t rsa -P '' -f ~/.ssh/id_rsa
```

Daftarkan kunci publik agar dipercaya oleh mesin itu sendiri:

```bash
cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
chmod 0600 ~/.ssh/authorized_keys
```

Uji coba — perintah berikut **seharusnya langsung masuk tanpa meminta password**:

```bash
ssh localhost
```

Jika berhasil masuk (terlihat perubahan prompt terminal), ketik `exit` untuk keluar dari sesi SSH tersebut dan kembali ke terminal biasa.

```bash
exit
```

> Jika masih diminta password, ulangi langkah `ssh-keygen` dan `cat ... >> authorized_keys` di atas dengan teliti — kesalahan paling umum adalah *typo* pada path `~/.ssh/`.


---
## 3.5 Unduh dan Ekstrak Apache Hadoop 3.4.3

**Langkah-langkah (jalankan di Terminal):**

1. Arahkan ke home directory:

```bash
cd ~
```

2. Unduh Hadoop 3.4.3 (berukuran sekitar 700-900 MB, proses ini dapat memakan waktu beberapa menit):

```bash
wget https://downloads.apache.org/hadoop/common/hadoop-3.4.3/hadoop-3.4.3.tar.gz
```

> Jika link tersebut tidak aktif (Apache sesekali memperbarui mirror), kunjungi **https://hadoop.apache.org/releases.html** di browser, cari baris **3.4.3**, klik **binary**, salin link download nya, lalu ganti URL pada perintah `wget` di atas.

3. Ekstrak berkas `.tar.gz` yang telah diunduh:

```bash
tar -xzvf hadoop-3.4.3.tar.gz
```

4. Ganti nama folder hasil ekstraksi agar lebih ringkas diketik:

```bash
mv hadoop-3.4.3 hadoop
```

5. Verifikasi folder Hadoop sudah ada:

```bash
ls ~/hadoop
```

Anda seharusnya melihat folder-folder seperti `bin`, `sbin`, `etc`, `lib`, dan lainnya.

---

## 3.6 Konfigurasi Environment Variables

Agar perintah Hadoop (`hdfs`, `start-dfs.sh`, dst.) dapat dipanggil dari mana saja tanpa mengetik path lengkap, tambahkan konfigurasi berikut ke berkas `~/.bashrc`.

**Buka berkas `.bashrc` menggunakan text editor nano:**

```bash
nano ~/.bashrc
```

**Gulir ke bagian paling bawah berkas, lalu tambahkan baris-baris berikut:**

```bash
# Konfigurasi Hadoop — Praktikum Big Data
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
export HADOOP_HOME=$HOME/hadoop
export HADOOP_INSTALL=$HADOOP_HOME
export HADOOP_MAPRED_HOME=$HADOOP_HOME
export HADOOP_COMMON_HOME=$HADOOP_HOME
export HADOOP_HDFS_HOME=$HADOOP_HOME
export YARN_HOME=$HADOOP_HOME
export HADOOP_COMMON_LIB_NATIVE_DIR=$HADOOP_HOME/lib/native
export PATH=$PATH:$HADOOP_HOME/sbin:$HADOOP_HOME/bin:$JAVA_HOME/bin
export HADOOP_OPTS="-Djava.library.path=$HADOOP_HOME/lib/native"
```

**Simpan dan keluar dari nano:** tekan `Ctrl + O` (huruf O, bukan angka 0) lalu **Enter** untuk menyimpan, kemudian `Ctrl + X` untuk keluar.

**Muat ulang konfigurasi** agar berlaku pada sesi terminal saat ini:

```bash
source ~/.bashrc
```

Verifikasi environment variable terbaca dengan benar:

```bash
echo $HADOOP_HOME
hadoop version
```

Perintah kedua seharusnya menampilkan informasi versi Hadoop 3.4.3 beserta detail build-nya — jika muncul error `command not found`, ulangi kembali langkah 3.6 dengan teliti (kemungkinan ada *typo* pada path).


---
## 3.7 Konfigurasi Berkas Hadoop

Ada 5 berkas konfigurasi yang perlu diedit, seluruhnya berada di dalam folder `~/hadoop/etc/hadoop/`.

### a) hadoop-env.sh — memberi tahu Hadoop lokasi Java

```bash
nano ~/hadoop/etc/hadoop/hadoop-env.sh
```

Cari baris yang bertuliskan `export JAVA_HOME=`, lalu ubah (atau tambahkan jika tidak ada) menjadi:

```bash
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
```

Simpan (`Ctrl+O`, Enter) dan keluar (`Ctrl+X`).

### b) core-site.xml — konfigurasi inti (alamat HDFS)

```bash
nano ~/hadoop/etc/hadoop/core-site.xml
```

Hapus seluruh isi berkas (gunakan `Ctrl+K` berulang kali untuk menghapus baris demi baris jika perlu), lalu ganti dengan:

```xml
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>
```

### c) hdfs-site.xml — konfigurasi HDFS

Terlebih dahulu, jalankan perintah berikut **di Terminal (bukan di dalam nano)** untuk mengetahui path *home directory* kalian secara pasti:

```bash
echo $HOME
```

Catat hasilnya (contoh: `/home/mahasiswa`), lalu buat folder penyimpanan data HDFS:

```bash
mkdir -p ~/hadoopdata/hdfs/namenode
mkdir -p ~/hadoopdata/hdfs/datanode
```

Sekarang edit berkas konfigurasinya:

```bash
nano ~/hadoop/etc/hadoop/hdfs-site.xml
```

Ganti isinya menjadi (**ganti `/home/mahasiswa` dengan hasil `echo $HOME` milik anda sendiri**):

```xml
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>
    <property>
        <name>dfs.namenode.name.dir</name>
        <value>/home/mahasiswa/hadoopdata/hdfs/namenode</value>
    </property>
    <property>
        <name>dfs.datanode.data.dir</name>
        <value>/home/mahasiswa/hadoopdata/hdfs/datanode</value>
    </property>
</configuration>
```

> **Mengapa `dfs.replication` bernilai 1?** Replikasi adalah jumlah salinan setiap blok data yang disimpan di DataNode berbeda untuk ketahanan terhadap kegagalan. Karena kita hanya memiliki **satu** DataNode (mode single-node), nilai replikasi wajib **1** — pada cluster produksi real di industri dengan banyak node, nilai umum yang dipakai adalah **3**.

### d) mapred-site.xml — konfigurasi MapReduce

```bash
nano ~/hadoop/etc/hadoop/mapred-site.xml
```

Isi dengan:

```xml
<configuration>
    <property>
        <name>mapreduce.framework.name</name>
        <value>yarn</value>
    </property>
</configuration>
```

### e) yarn-site.xml — konfigurasi YARN

```bash
nano ~/hadoop/etc/hadoop/yarn-site.xml
```

Isi dengan:

```xml
<configuration>
    <property>
        <name>yarn.nodemanager.aux-services</name>
        <value>mapreduce_shuffle</value>
    </property>
</configuration>
```

Setelah kelima berkas di atas tersimpan, konfigurasi Hadoop untuk mode pseudo-distributed sudah lengkap.


---
## 3.8 Format HDFS NameNode

Sebelum HDFS dapat digunakan untuk **pertama kali**, NameNode wajib diformat (mirip memformat hard disk baru). **Perintah ini hanya dijalankan SATU KALI** — menjalankannya ulang di kemudian hari akan **menghapus seluruh data** yang tersimpan di HDFS.

```bash
hdfs namenode -format
```

Perhatikan output di terminal — cari baris yang bertuliskan `Storage directory ... has been successfully formatted` sebagai tanda keberhasilan.

---

## 3.9 Menjalankan Hadoop Daemons

Jalankan proses HDFS (NameNode, DataNode, SecondaryNameNode):

```bash
start-dfs.sh
```

Lanjutkan dengan menjalankan proses YARN (ResourceManager, NodeManager):

```bash
start-yarn.sh
```

Kedua perintah di atas akan menampilkan proses *starting* untuk masing-masing daemon tanpa perlu memasukkan password (berkat konfigurasi SSH pada Sub-bab 3.4).

**Verifikasi seluruh daemon berjalan** menggunakan perintah `jps` (Java Process Status):

```bash
jps
```

Output yang diharapkan berupa **5 proses** berikut (urutan dan nomor PID dapat berbeda):

```
12345 NameNode
12456 DataNode
12567 SecondaryNameNode
12678 ResourceManager
12789 NodeManager
12890 Jps
```

> **Jika ada proses yang tidak muncul** (misalnya DataNode tidak jalan), Hadoop kemungkinan mengalami error saat startup. Periksa log di folder `~/hadoop/logs/` (cari berkas `.log` dengan nama sesuai daemon yang bermasalah, lihat baris `ERROR` di bagian akhir berkas) untuk mendiagnosis penyebabnya — penyebab tersering adalah kesalahan path pada `hdfs-site.xml` atau `JAVA_HOME` yang salah.

---

## 3.10 Mengakses Web UI Hadoop

Hadoop menyediakan dashboard berbasis web untuk memantau status cluster. Buka **Firefox di dalam VM Ubuntu**, lalu kunjungi:

| Dashboard | Alamat | Kegunaan |
|---|---|---|
| **NameNode UI** | http://localhost:9870 | Melihat status HDFS, kapasitas, daftar DataNode, menjelajah berkas |
| **ResourceManager UI** | http://localhost:8088 | Melihat status YARN dan daftar aplikasi/job yang berjalan |

Jika kedua halaman tersebut berhasil terbuka dan menampilkan dashboard (bukan pesan error "connection refused"), maka instalasi Hadoop anda **berhasil sepenuhnya**.

**Perintah untuk menghentikan Hadoop** (gunakan setelah selesai praktikum, sebelum mematikan VM):

```bash
stop-yarn.sh
stop-dfs.sh
```

> **Kebiasaan baik:** selalu jalankan `stop-yarn.sh` dan `stop-dfs.sh` sebelum mematikan VM, dan jalankan kembali `start-dfs.sh` & `start-yarn.sh` setiap kali membuka VM baru (namenode **tidak perlu** diformat ulang — format hanya sekali seumur hidup HDFS, kecuali kalian sengaja ingin mengosongkan seluruh data).


---
## Melanjutkan di Jupyter Notebook

Sampai di sini, seluruh instalasi selesai dan Hadoop sedang berjalan. Sekarang buka **Terminal baru**, jalankan urutan perintah berikut untuk masuk ke environment `bigdata` dan menjalankan Jupyter (persis seperti Pertemuan 2):

```bash
conda activate bigdata
cd ~/praktikum-bigdata
jupyter notebook
```

Buka file modul **Pertemuan 3** ini di dalam Jupyter yang baru terbuka, lalu lanjutkan ke Sub-bab 3.11 di bawah ini — mulai dari sini seluruh instruksi berupa **code cell** yang dapat langsung dijalankan.

> **Catatan teknis:** Karena `PATH` menuju perintah `hdfs` sudah kita tambahkan ke `~/.bashrc` pada Sub-bab 3.6, dan Jupyter dijalankan dari terminal yang sama, seluruh perintah Hadoop otomatis dapat dipanggil dari dalam notebook menggunakan tanda seru (`!`) di awal baris.


In [1]:
# Sel tes: memastikan perintah HDFS dapat dipanggil dari dalam Jupyter
!hdfs version

Hadoop 3.4.3
Source code repository https://github.com/apache/hadoop.git -r 9d50c6884666e794e45102260a4017bb31802e1b
Compiled by stevel on 2026-02-13T14:23Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum 2331238c4c2929e66645316a32a8613
This command was run using /home/irkham/hadoop/share/hadoop/common/hadoop-common-3.4.3.jar


Jika muncul output informasi versi Hadoop (bukan pesan error `command not found`), berarti integrasi Jupyter dengan Hadoop berhasil. Lanjutkan ke hands-on berikut.

---

## 3.11 Hands-On: Perintah Dasar HDFS

Perintah HDFS memiliki pola umum: **`hdfs dfs -[perintah] [argumen]`** — mirip perintah Linux biasa (`ls`, `mkdir`, `cp`), namun beroperasi pada sistem berkas terdistribusi HDFS, bukan disk lokal.

### 3.11.1 Membuat Direktori di HDFS

In [3]:
# Membuat direktori kerja pribadi di HDFS
# HDFS memiliki struktur folder sendiri, terpisah dari folder Linux biasa
!hdfs dfs -mkdir -p /user/irkham/praktikum

# Menampilkan isi direktori root HDFS untuk verifikasi
!hdfs dfs -ls /user/irkham

Found 1 items
drwxr-xr-x   - irkham supergroup          0 2026-09-05 17:11 /user/irkham/praktikum


### 3.11.2 Upload Berkas dari Lokal ke HDFS (`put`)

In [4]:
# Membuat berkas teks sederhana di disk lokal VM (bukan HDFS) sebagai contoh
with open("contoh_lokal.txt", "w") as f:
    f.write("Halo dari Praktikum Big Data!\n")
    f.write("Berkas ini awalnya tersimpan di disk lokal VM.\n")

print("Berkas lokal berhasil dibuat.")
!ls -la contoh_lokal.txt

Berkas lokal berhasil dibuat.
-rw-rw-r-- 1 irkham irkham 77 ก.ย.   5 17:12 contoh_lokal.txt


In [5]:
# Upload (put) berkas lokal ke dalam HDFS
!hdfs dfs -put contoh_lokal.txt /user/irkham/praktikum/

# Verifikasi berkas sudah ada di HDFS
!hdfs dfs -ls /user/irkham/praktikum

Found 1 items
-rw-r--r--   1 irkham supergroup         77 2026-09-05 17:13 /user/irkham/praktikum/contoh_lokal.txt


### 3.11.3 Melihat Isi Berkas di HDFS (`cat`)

In [6]:
!hdfs dfs -cat /user/irkham/praktikum/contoh_lokal.txt

Halo dari Praktikum Big Data!
Berkas ini awalnya tersimpan di disk lokal VM.


### 3.11.4 Download Berkas dari HDFS ke Lokal (`get`)

In [7]:
# Mengunduh (get) berkas dari HDFS kembali ke disk lokal dengan nama berbeda
!hdfs dfs -get /user/irkham/praktikum/contoh_lokal.txt hasil_unduh_dari_hdfs.txt

# Verifikasi isinya sama
!cat hasil_unduh_dari_hdfs.txt

Halo dari Praktikum Big Data!
Berkas ini awalnya tersimpan di disk lokal VM.


### 3.11.5 Informasi Ukuran dan Jumlah Berkas

In [8]:
# -du: menampilkan ukuran penggunaan disk di HDFS
!hdfs dfs -du -h /user/irkham/praktikum

# -count: menampilkan jumlah direktori, berkas, dan total ukuran
!hdfs dfs -count /user/irkham/praktikum

77  77  /user/irkham/praktikum/contoh_lokal.txt
           1            1                 77 /user/irkham/praktikum


### 3.11.6 Menghapus Berkas/Direktori di HDFS (`rm`)

In [9]:
# Menghapus satu berkas di HDFS
!hdfs dfs -rm /user/irkham/praktikum/contoh_lokal.txt

# Verifikasi berkas sudah terhapus (direktori seharusnya kosong sekarang)
!hdfs dfs -ls /user/irkham/praktikum

Deleted /user/irkham/praktikum/contoh_lokal.txt


### Ringkasan Perintah HDFS yang Telah Dipelajari

| Perintah | Fungsi |
|---|---|
| `hdfs dfs -mkdir -p [path]` | Membuat direktori (dengan `-p`, otomatis membuat parent folder jika belum ada) |
| `hdfs dfs -ls [path]` | Menampilkan isi direktori |
| `hdfs dfs -put [lokal] [hdfs]` | Mengunggah berkas dari disk lokal ke HDFS |
| `hdfs dfs -get [hdfs] [lokal]` | Mengunduh berkas dari HDFS ke disk lokal |
| `hdfs dfs -cat [path]` | Menampilkan isi berkas teks di HDFS |
| `hdfs dfs -du -h [path]` | Menampilkan ukuran penggunaan disk (format *human-readable*) |
| `hdfs dfs -count [path]` | Menghitung jumlah direktori/berkas dan total ukuran |
| `hdfs dfs -rm [path]` | Menghapus berkas (tambahkan `-r` untuk menghapus direktori beserta isinya) |


---
## Latihan Mandiri

Kerjakan latihan berikut menggunakan cell kode kosong di bawah masing-masing soal.

**Soal 1.** Buatlah direktori baru di HDFS dengan path `/user/mahasiswa/latihan3`.

In [10]:
# Jawaban Soal 1 di sini
!hdfs dfs -mkdir -p /user/mahasiswa/latihan3
!hdfs dfs -ls /user/mahasiswa/

Found 2 items
drwxr-xr-x   - irkham supergroup          0 2026-09-05 17:21 /user/mahasiswa/latihan3
drwxr-xr-x   - irkham supergroup          0 2026-09-05 17:11 /user/mahasiswa/praktikum


**Soal 2.** Buatlah sebuah berkas teks lokal berisi biodata singkat kalian (nama, NIM, hobi — minimal 3 baris), lalu upload ke direktori `/user/mahasiswa/latihan3` yang baru dibuat pada Soal 1.

In [12]:
# Jawaban Soal 2 di sini
with open("biodata_singkat.txt", "w") as f:
    f.write("Nama: Zuyina M. Irkham\n")
    f.write("NPM: 2505060041\n")
    f.write("Hobi: Bermain Bulu Tangkis\n")

!hdfs dfs -put biodata_singkat.txt /user/mahasiswa/latihan3/

**Soal 3.** Tampilkan isi berkas biodata kalian langsung dari HDFS (tanpa download terlebih dahulu), menggunakan perintah `cat`.

In [13]:
# Jawaban Soal 3 di sini
!hdfs dfs -cat /user/mahasiswa/latihan3/biodata_singkat.txt

Nama: Zuyina M. Irkham
NPM: 2505060041
Hobi: Bermain Bulu Tangkis


**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat, jelaskan menurut pemahaman kalian sendiri: apa perbedaan mendasar antara menyimpan berkas di **disk lokal VM** (seperti `contoh_lokal.txt` tadi) dengan menyimpan di **HDFS**? Tuliskan jawaban pada cell markdown di bawah ini.

*(Tulis jawaban di sini)*
Untuk penyimpanan di disk lokal VM hanya menyimpan di 1 komputer aja jadi jika komputernya rusak maka data bisa hilang.
Untuk penyimpanan HDFS itu membagi berkas menjadi blok-blok kecil dan didistribusikan beserta replikasinya ke banyak komputer sehingga jauh lebih aman,handal dan cocok untuk mengatasi skala big data

---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

### Konteks / Skenario

Masih ingat platform e-commerce tempat kalian magang pada Tugas Mandiri Pertemuan 2? Bulan ini, tim engineering mengumumkan bahwa perusahaan mulai **bermigrasi ke infrastruktur Big Data** — seluruh data transaksi dari cabang-cabang di berbagai kota tidak lagi disimpan sebagai berkas CSV lepas di laptop analyst, melainkan akan disimpan secara terpusat di **HDFS**, sebagai fondasi sebelum diproses lebih lanjut menggunakan tools big data di pertemuan-pertemuan mendatang. Sebagai data analyst junior, kalian ditugaskan menjadi orang pertama yang mencoba alur kerja ini secara menyeluruh: dari data mentah di tiga kota berbeda, hingga tersimpan rapi di HDFS dan siap dianalisis kembali.

### Menyiapkan Dataset

Jalankan cell kode berikut **di dalam modul ini terlebih dahulu** untuk menghasilkan **tiga berkas CSV terpisah** (mensimulasikan data transaksi dari tiga cabang kota berbeda selama Agustus 2026).

In [1]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


### Instruksi Pengerjaan

Buat **notebook BARU** bernama **`Tugas3_[NIM]_[Nama Lengkap].ipynb`**, letakkan di folder `~/praktikum-bigdata` yang sama dengan ketiga berkas CSV di atas, lalu kerjakan seluruh bagian **A sampai E** berikut secara berurutan.

---

**A. Membangun Struktur Direktori HDFS** *(bobot 15%)*

Buatlah struktur direktori berikut di HDFS:
```
/user/[username]/ecommerce/raw
/user/[username]/ecommerce/processed
```
Tampilkan hasilnya untuk membuktikan struktur sudah terbentuk.

**B. Mengunggah Data Mentah ke HDFS** *(bobot 15%)*

Unggah **ketiga berkas CSV** (`transaksi_magelang.csv`, `transaksi_yogyakarta.csv`, `transaksi_semarang.csv`) ke direktori `/user/[username]/ecommerce/raw` di HDFS. Tampilkan isi direktori tersebut sebagai bukti ketiga berkas berhasil terunggah, lengkap dengan ukurannya.

**C. Membaca Kembali dan Menggabungkan Data dari HDFS** *(bobot 25%)*

Menggunakan Python, baca kembali **ketiga berkas dari HDFS** (bukan dari disk lokal!). Gabungkan ketiganya menjadi **satu DataFrame**, lalu buktikan hasil gabungan berisi transaksi dari ketiga kota (tampilkan `value_counts()` pada kolom `kota`).

**D. Mengolah dan Download Hasil ke Direktori `processed`** *(bobot 25%)*

Pada DataFrame gabungan hasil bagian C:
1. Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`).
2. Buatlah **satu tabel ringkasan** (`groupby`) yang menampilkan total pendapatan per kota **dan** per kategori sekaligus.
3. Simpan **dua** hasil berikut sebagai berkas CSV baru di disk lokal, lalu unggah keduanya ke `/user/[username]/ecommerce/processed` di HDFS:
   - `data_gabungan_bersih.csv` (seluruh data ketiga kota yang telah digabung + kolom `total_pendapatan`)
   - `ringkasan_kota_kategori.csv` (tabel ringkasan dari langkah 2)

**E. Dokumentasi dan Refleksi** *(bobot 20%)*

Sertakan pada notebook anda:
1. **Screenshot** halaman NameNode Web UI (http://localhost:9870, menu *Utilities → Browse the file system*) yang menunjukkan struktur folder `/user/[username]/ecommerce/` beserta isinya (paste gambar ke dalam markdown cell, atau lampirkan terpisah jika format pengumpulan berupa arsip/zip.
2. Tulisan reflektif (**minimal 100 kata**) pada markdown cell yang menjawab: *Apa keuntungan menyimpan data mentah (raw) terpisah dari data olahan (processed) di HDFS, dibandingkan menyimpan semuanya bercampur dalam satu folder?*

---